In [4]:
%pip install pythae

  Using cached pythae-0.1.2-py3-none-any.whl.metadata (52 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached imageio-2.37.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached dataclasses-0.6-py3-none-any.whl.metadata (3.0 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached pythae-0.1.2-py3-none-any.whl (235 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached dataclasses-0.6-py3-none-any.whl (14 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 14.6 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   --- ------------------------------------ 2.9/37.3 MB 14.7 MB/s eta 0:00:03
   ----- ---


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

class ImageFolderFlat(Dataset):
    def __init__(self, root, transform=None):
        self.paths = [
            os.path.join(root, f) for f in os.listdir(root)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
        ]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")  # use "L" for grayscale
        if self.transform:
            img = self.transform(img)
        return img, 0  # dummy label so it works with the training loop unchanged

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

dataset = ImageFolderFlat("./data/pokemon_data/train", transform=transform)
loader  = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)

In [ ]:
from pythae.models import AE, AEConfig
from pythae.trainers import BaseTrainer, BaseTrainerConfig

model_config = AEConfig(input_dim=(3, 64, 64), latent_dim=16)
model = AE(model_config=model_config)

trainer_config = BaseTrainerConfig(num_epochs=10, learning_rate=1e-3, batch_size=16)
trainer = BaseTrainer(model=model, train_dataset=loader.dataset, training_config=trainer_config)
trainer.train()

# Extract embeddings
encoded = model.encoder(image_tensor)  # → (N, 16)

! No eval dataset provided ! -> keeping best model on train.



ModelError: Error when calling forward method from model. Potential issues: 
 - Wrong model architecture -> check encoder, decoder and metric architecture if you provide yours 
 - The data input dimension provided is wrong -> when no encoder, decoder or metric provided, a network is built automatically but requires the shape of the flatten input data.
Exception raised: <class 'TypeError'> with message: list indices must be integers or slices, not str

In [1]:
class ImageFolderFlat(Dataset):
    def __init__(self, root, transform=None):
        self.paths = [
            os.path.join(root, f) for f in os.listdir(root)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
        ]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")  # use "L" for grayscale
        if self.transform:
            img = self.transform(img)
        return img, 0  # dummy label so it works with the training loop unchanged

NameError: name 'Dataset' is not defined

In [ ]:
import torch
from torch import optim, nn

def train(model, loader, epochs=30, lr=1e-3, device="cpu"):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    #criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        total = 0
        for imgs, _ in loader:
            imgs = imgs.to(device)
            optimizer.zero_grad()
            recon = model(imgs)
            loss = recon.loss
            loss.backward()
            optimizer.step()
            total += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}  Loss: {total/len(loader):.4f}")


# ── Embeddings ───────────────────────────────────────────────────────────────

def get_embeddings(model, loader, device="cpu"):
    model.eval()
    all_z = []
    with torch.no_grad():
        for imgs, _ in loader:
            z = model.encoder(imgs.to(device)).embedding
            all_z.append(z)
    return torch.cat(all_z).numpy()


In [52]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch

from pythae.models import AE, AEConfig
from pythae.trainers import BaseTrainer, BaseTrainerConfig

# 1. Dataset & loader
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

dataset = ImageFolderFlat(root="./Images/0/", transform=transform)

n_val = int(0.1 * len(dataset))
train_ds, val_ds = random_split(dataset, [len(dataset) - n_val, n_val])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

# 2. Model
device = "cuda" if torch.cuda.is_available() else "cpu"

model_config = AEConfig(input_dim=(3, 64, 64), latent_dim=16)
model = AE(model_config=model_config)

# 3. Train — pass train_loader directly
train(model, train_loader, epochs=10, device=device)


Epoch 5/10  Loss: 296.8281
Epoch 10/10  Loss: 198.6167


In [60]:
train_embeddings = get_embeddings(model, train_loader, device=device)

tensor([[-2.2557e+01,  1.1277e+01, -2.2668e+01, -2.4921e+01, -3.1136e+00,
         -1.7697e+01,  1.4201e+01,  9.0132e+00, -1.1621e+01, -2.3238e+01,
         -1.7882e+01, -2.3345e+00, -1.2856e+01,  2.2428e+01, -8.8730e+00,
          1.7489e+01],
        [-1.9246e+01,  8.4039e+00, -1.8129e+01, -1.9604e+01, -1.8714e+00,
         -1.4575e+01,  1.1701e+01,  6.0370e+00, -9.4048e+00, -1.8893e+01,
         -1.4412e+01, -1.2747e+00, -1.0268e+01,  1.8031e+01, -6.1331e+00,
          1.4512e+01],
        [ 4.5746e+01, -7.5507e+01,  9.2790e+01,  4.9842e+01,  4.7459e+01,
          3.3153e+01, -1.0636e+02, -3.0661e+01,  8.5546e+01, -9.1041e+00,
          2.8634e+01, -5.2024e+01,  3.8680e+01, -3.9174e+01,  6.7605e+01,
          5.0027e+01],
        [-1.3733e+01,  4.6473e+00, -1.1966e+01, -1.2724e+01, -5.2182e-01,
         -1.0027e+01,  7.1283e+00,  2.6782e+00, -5.6253e+00, -1.2918e+01,
         -1.0411e+01, -1.2094e+00, -6.4822e+00,  1.2163e+01, -2.6608e+00,
          1.1179e+01],
        [-7.4177e+00

In [46]:
train_embeddings.shape

(223, 16)

In [72]:
import numpy as np

recons = np.array(())
for imgs, _ in train_loader:
    imgs = imgs.to(device)
    recon = model.forward(imgs)
    recon = recon.recon_x
    recon = torch.einsum('nchw->hwc', recon).detach().cpu()

print(recon.shape)

torch.Size([64, 64, 3])


In [71]:
import matplotlib.pyplot as plt

def show_image(image, title=''):
    # image is [H, W, 3]
    assert image.shape[2] == 3
    plt.imshow(torch.clip((image * imagenet_std + imagenet_mean) * 255, 0, 255).int())
    plt.title(title, fontsize=16)
    plt.axis('off')
    return

In [ ]:
imagenet_mean = np.array(feature_extractor.image_mean)
imagenet_std = np.array(feature_extractor.image_std)

In [74]:
model

AE(
  (decoder): Decoder_AE_MLP(
    (layers): ModuleList(
      (0): Sequential(
        (0): Linear(in_features=16, out_features=512, bias=True)
        (1): ReLU()
      )
      (1): Sequential(
        (0): Linear(in_features=512, out_features=12288, bias=True)
        (1): Sigmoid()
      )
    )
  )
  (encoder): Encoder_AE_MLP(
    (layers): ModuleList(
      (0): Sequential(
        (0): Linear(in_features=12288, out_features=512, bias=True)
        (1): ReLU()
      )
    )
    (embedding): Linear(in_features=512, out_features=16, bias=True)
  )
)

In [73]:
show_image(recon)

NameError: name 'imagenet_std' is not defined